# 🧪 Antigravity Research Lab

Use this notebook to train your ML models interactively.
Any changes you make to the model's 'Brain' (Checkpoints) will be picked up by the Live Trading App.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# Add current directory to path so we can import our modules
sys.path.append(os.getcwd())

from ml_engine import PredictiveAlphaEngine
from external_data import fetch_market_data_pool
from constants import FUTURES_UNIVERSE

## 1. Load Data
Fetching latest market data from Yahoo Finance...

In [ ]:
# Select Assets
ASSETS = FUTURES_UNIVERSE[:10] # Top 10 for quick testing
print(f"Fetching data for {len(ASSETS)} assets: {ASSETS}")

# Fetch Data
data_pool = fetch_market_data_pool(ASSETS)

# Convert to DataFrame (Dates x Assets)
df_dict = {}
for asset, info in data_pool.items():
    if info['data']:
        ts = pd.DataFrame(info['data'])
        ts['date'] = pd.to_datetime(ts['date'])
        ts.set_index('date', inplace=True)
        df_dict[asset] = ts['price_index']

df_prices = pd.DataFrame(df_dict).dropna()
print(f"Loaded {len(df_prices)} days of price data.")
df_prices.tail()

## 2. Initialize Neural Brain
Load existing checkpoint if available.

In [ ]:
CHECKPOINT_PATH = "ml_checkpoint.pkl"

engine = PredictiveAlphaEngine(ASSETS)
if engine.load_checkpoint(CHECKPOINT_PATH):
    print("✅ Loaded existing brain!")
else:
    print("❌ Starting fresh brain.")

## 3. Train (Isolate Epoch)
Run this cell multiple times to train more epochs.

In [ ]:
# Extract features and update models for the entire history
print("Training...")

errors = []

# Iterate through history
window = 30
for t in range(window, len(df_prices)-1):
    
    # Get prices up to t
    # In 'update', we pretend we are at time t, and we just observed price[t].
    # We want to learn from the transition t-1 -> t
    
    start_idx = max(0, t-15)
    # Window for features (needs 10-12 days before today)
    # We want to predict returns at t (observed today).
    # Features are from t-1.
    
    # This logic matches Backtester.run essentially
    target_ret = df_prices.iloc[t].pct_change().fillna(0)
    
    # Only train 1 asset for visualization demo
    asset = ASSETS[0]
    
    # Get historical slice ending at t-1 for features
    # slice [t-12 : t] (exclusive t) -> ends at t-1
    hist_slice = df_prices[asset].values[max(0, t-12) : t]
    
    if len(hist_slice) > 10:
        feats = engine._extract_features(hist_slice)
        actual_y = df_prices[asset].iloc[t] / df_prices[asset].iloc[t-1] - 1.0
        
        err = engine.models[asset].update(feats, actual_y)
        errors.append(err)

print(f"Training complete. Processed {len(df_prices)} vectors.")

# Plot Error Convergence
plt.figure(figsize=(10, 4))
plt.plot(errors, alpha=0.5)
plt.title(f"Learning Curve (Prediction Error) for {ASSETS[0]}")
plt.xlabel("Time Steps")
plt.ylabel("Error")
plt.grid(True)
plt.show()

## 4. Save Checkpoint
Commit your changes to the brain.

In [ ]:
engine.save_checkpoint(CHECKPOINT_PATH)
print(f"Saved brain to {CHECKPOINT_PATH}")